# Evaluation AutoEncoders

This notebook summarizes the evaluation results of the trained models.

## 1. Import Libraries

In [ ]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score,accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import tqdm
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
import pyarrow.parquet as pq
from ipywidgets import interact, FloatSlider

# Add the project root to the Python path to allow importing modelPipelinesTL
# Assuming the notebook is in the root of the 'tool-wear-domain-adaption' project
project_root = os.path.abspath('.')
if project_root not in sys.path:
    sys.path.append(project_root)

from modelPipelinesTL import AutoencoderPipeline

## 2. Define Paths and Load Configurations

In [ ]:
# Define the parent directory for experiments
output_parent_dir = os.path.abspath(os.path.expanduser(
    #'/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/tool-wear-domain-adaption/experiments/nature_milling_autoencoder'
    '/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/tool-wear-domain-adaption/experiments/phm_milling_autoencoder'
))

# Load configs and model paths from all experiments
experiment_data = []
config_dir = os.path.join(output_parent_dir, 'configs')
model_dir = os.path.join(output_parent_dir, 'models')

if os.path.exists(model_dir) and os.path.exists(config_dir):
    # Create a dictionary mapping timestamps to config file paths
    config_files = {
        f.split('config_')[-1].replace('.yaml', ''): os.path.join(config_dir, f)
        for f in os.listdir(config_dir) if f.endswith('.yaml')
    }
    
    # Find all model files and match them with configs
    model_files = [f for f in os.listdir(model_dir) if f.endswith('.keras')]
    for model_filename in model_files:
        model_path = os.path.join(model_dir, model_filename)
        # Extract timestamp from filename, e.g., 1d_conv_model_20251009_090843.keras
        timestamp = model_filename.split('model_')[-1].replace('.keras', '')
        
        if timestamp in config_files:
            config_path = config_files[timestamp]
            with open(config_path, 'r') as f:
                config = yaml.safe_load(f)
            
            experiment_data.append({
                'config_path': config_path,
                'model_path': model_path,
                'config': config,
                'train_caseIDs': config.get('train_caseIDs', []),
                'test_caseIDs': config.get('test_caseIDs', [])
            })

# Create a DataFrame to display the experiment configurations
if experiment_data:
    experiments_df = pd.DataFrame([{
        'Train Cases': d['train_caseIDs'],
        'Test Cases': d['test_caseIDs'],
        'Model Path': os.path.basename(d['model_path']),
        'Config Path': os.path.basename(d['config_path'])
    } for d in experiment_data])
else:
    experiments_df = pd.DataFrame()

# Sort by train and test cases for consistent order
if not experiments_df.empty:
    # Convert list to tuple for sorting
    experiments_df['Train Cases'] = experiments_df['Train Cases'].apply(tuple)
    experiments_df['Test Cases'] = experiments_df['Test Cases'].apply(tuple)
    experiments_df = experiments_df.sort_values(by=['Train Cases', 'Test Cases']).reset_index(drop=True)

experiments_df.head()

## 3. Load and Prepare Test Data

In [ ]:
signal_columns = ['vib_spindle']

In [ ]:
# merge single experiment results into one dataframe
results_df = pd.DataFrame()
# The single files are in the logs folder named: evaluation_results_1d_conv_$timestamp.csv but just get every csv file in the folder
# They all have the same header so we can just concatenate them
logs_dir = os.path.join(output_parent_dir, 'logs')
if os.path.exists(logs_dir):
    log_files = [f for f in os.listdir(logs_dir) if f.endswith('.csv')]
    for log_file in log_files:
        log_path = os.path.join(logs_dir, log_file)
        df = pd.read_csv(log_path)
        results_df = pd.concat([results_df, df], ignore_index=True)

## 4. Evaluate Models and Collect Results

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
from IPython.display import display, Markdown
import ast
import re

def parse_string_to_list(s):
    if isinstance(s, list):
        return s
    if not isinstance(s, str):
        return s
    try:
        # First, try ast.literal_eval for standard list formats
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        # If that fails, handle numpy-style string arrays (e.g., "[1. 2. 3.]")
        s = s.strip('[]')
        # Replace newline characters and use regex to find all numbers
        s = s.replace('\n', ' ')
        # This regex finds floating point and integer numbers
        numbers = re.findall(r'[-+]?\d*\.\d+|\d+', s)
        return [float(n) for n in numbers]

results_df = pd.DataFrame()
summary_path = os.path.join(output_parent_dir, 'evaluation_summary.csv')
is_autoencoder = 'autoenc' in output_parent_dir

if os.path.exists(summary_path):
    print(f"Loading existing evaluation summary from {summary_path}")
    results_df = pd.read_csv(summary_path)
    
    # Define columns that might need conversion from string to list/array
    cols_to_convert = {
        'Train Cases', 'Test Cases', 'confusion_matrix', 
        'y_true_classes', 'y_pred_classes', 
        'reconstruction_error', 'y_true'
    }
    
    for col in results_df.columns:
        if col in cols_to_convert:
            # Check if the column is not empty and contains string data before converting
            if not results_df[col].empty and isinstance(results_df[col].iloc[0], str):
                results_df[col] = results_df[col].apply(parse_string_to_list)

else:
    print("No existing summary file found. Running evaluation...")
    results = []
    for exp_data in tqdm.tqdm(experiment_data, desc="Evaluating models"):
        
        # Add the config_path to the config dictionary itself for use within the pipeline
        exp_data['config']['config_path'] = exp_data['config_path']
        
        # Initialize the pipeline with the experiment's config
        pipeline = AutoencoderPipeline(exp_data['config'])
        
        # Load the model
        pipeline.model = tf.keras.models.load_model(exp_data['model_path'])
        
        # Prepare the data using the pipeline's method
        pipeline.prepare_data()
        
        # The pipeline's evalModel method calculates and saves results, but we want to collect them here.
        # So, we'll replicate the core logic of evalModel.
        reconstructions = pipeline.model.predict(pipeline.X_test)
        recon_error = np.mean(np.abs(reconstructions - pipeline.X_test), axis=(1, 2))
        
        results.append({
            'train_caseIDs': exp_data['train_caseIDs'],
            'test_caseIDs': exp_data['test_caseIDs'],
            'reconstruction_error': recon_error.tolist(),
            'y_true': pipeline.y_test.tolist()
        })

    # Create a DataFrame from the collected results
    results_df = pd.DataFrame(results) if results else pd.DataFrame()

# Process and display the DataFrame
if not results_df.empty:
    # Rename columns for clarity and consistency
    if is_autoencoder:
        results_df = results_df.rename(columns={'train_caseIDs': 'Train Cases', 'test_caseIDs': 'Test Cases'})
    else:
        results_df = results_df.rename(columns={
            'train_caseIDs': 'Train Cases', 'test_caseIDs': 'Test Cases',
            'accuracy': 'Accuracy', 'loss': 'Loss',
            'precision': 'Precision', 'recall': 'Recall',
            'f1_score': 'F1-Score',
            'y_true': 'y_true_classes', 'y_pred': 'y_pred_classes'
        })
        
        # Reorder columns to have metrics first, then the data arrays
        desired_columns = [
            'Train Cases', 'Test Cases', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'Loss',
            'confusion_matrix', 'y_true_classes', 'y_pred_classes'
        ]
        existing_desired_columns = [col for col in desired_columns if col in results_df.columns]
        results_df = results_df[existing_desired_columns]

    # Sort by train and test cases for consistent order
    if 'Train Cases' in results_df.columns:
        # Convert list to tuple for sorting
        results_df['Train Cases'] = results_df['Train Cases'].apply(lambda x: tuple(x) if isinstance(x, list) else x)
        results_df['Test Cases'] = results_df['Test Cases'].apply(lambda x: tuple(x) if isinstance(x, list) else x)
        results_df = results_df.sort_values(by=['Train Cases', 'Test Cases']).reset_index(drop=True)

    if not is_autoencoder:
        display(Markdown("### Overall Performance Metrics"))

display(results_df)

In [ ]:
# Save if not already performed during batch evaluation
if not os.path.exists(summary_path):
    results_df.to_csv(summary_path, index=False)
    print(f"Saved evaluation summary to {summary_path}")

## 11. Autoencoder Anomaly Detection Evaluation

In [ ]:
from ipywidgets import Dropdown, VBox, HBox, Output
from IPython.display import display

# This cell evaluates an autoencoder model. It calculates the reconstruction error
# and then uses an interactive slider to set a threshold for anomaly detection.

# Check if there is experiment data and if the model is an autoencoder
if 'autoenc' in output_parent_dir and not results_df.empty:
    
    # Create a dropdown to select an experiment
    experiment_options = [
        (f"Train: {row['Train Cases']}, Test: {row['Test Cases']}", idx)
        for idx, row in results_df.iterrows()
    ]
    
    experiment_selector = Dropdown(
        options=experiment_options,
        description='Experiment:',
        style={'description_width': 'initial'}
    )

    def run_evaluation_for_experiment(experiment_index):
        # Select the experiment from the results DataFrame
        autoenc_results = results_df.iloc[experiment_index]
        
        # Ensure recon_error and y_true are numpy arrays of numbers
        recon_error = np.array(autoenc_results['reconstruction_error'], dtype=np.float32)
        y_true = np.array(autoenc_results['y_true'], dtype=np.int32)
        
        # Get the true labels for worn and non-worn tools
        is_worn = y_true == 1
        
        # --- Pre-calculate metrics for the plot ---
        thresholds = np.linspace(np.min(recon_error), np.max(recon_error), num=200)
        accuracies, precisions, recalls, f1_scores = [], [], [], []

        for t in thresholds:
            y_pred = (recon_error > t).astype(int)
            accuracies.append(accuracy_score(y_true, y_pred))
            precisions.append(precision_score(y_true, y_pred, zero_division=0))
            recalls.append(recall_score(y_true, y_pred, zero_division=0))
            f1_scores.append(f1_score(y_true, y_pred, zero_division=0))

        # Create a summary plot of reconstruction errors
        plt.figure(figsize=(12, 6))
        plt.hist(recon_error[~is_worn], bins=50, alpha=0.5, label='Healthy (Class 0)')
        plt.hist(recon_error[is_worn], bins=50, alpha=0.5, label='Worn (Class 1)')
        plt.xlabel("Reconstruction Error (MAE)")
        plt.ylabel("Number of Samples")
        plt.title(f"Distribution of Reconstruction Errors for Experiment {experiment_index}")
        plt.legend()
        plt.show()

        def plot_metrics_with_threshold(threshold):
            # Plot the metrics
            plt.figure(figsize=(12, 7))
            plt.plot(thresholds, accuracies, label='Accuracy')
            plt.plot(thresholds, precisions, label='Precision')
            plt.plot(thresholds, recalls, label='Recall')
            plt.plot(thresholds, f1_scores, label='F1-Score', linewidth=2)
            
            # Add a vertical line for the selected threshold
            plt.axvline(x=threshold, color='r', linestyle='--', label=f'Selected Threshold ({threshold:.3f})')
            
            plt.xlabel("Threshold")
            plt.ylabel("Score")
            plt.title("Metrics vs. Reconstruction Error Threshold")
            plt.legend()
            plt.grid(True, which='both', linestyle='--', linewidth=0.5)
            plt.show()

            # Print the metrics for the selected threshold
            y_pred = (recon_error > threshold).astype(int)
            accuracy = np.mean(y_pred == y_true)
            precision = precision_score(y_true, y_pred, zero_division=0)
            recall = recall_score(y_true, y_pred, zero_division=0)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            
            print(f"Metrics for Threshold: {threshold:.4f}")
            print(f"Accuracy: {accuracy:.4f}")
            print(f"Precision: {precision:.4f}")
            print(f"Recall: {recall:.4f}")
            print(f"F1-Score: {f1:.4f}")

        # Interactive slider for the threshold
        interact(
            plot_metrics_with_threshold,
            threshold=FloatSlider(
                min=np.min(recon_error), 
                max=np.max(recon_error), 
                step=(np.max(recon_error) - np.min(recon_error)) / 200, 
                value=np.median(recon_error), 
                description='Recon. Threshold',
                continuous_update=False
            )
        );

    # Use interact to link the dropdown to the evaluation function
    interact(run_evaluation_for_experiment, experiment_index=experiment_selector)

else:
    print("This evaluation is designed for autoencoder models.")
    print("Please set 'output_parent_dir' to an autoencoder experiment directory, e.g., '.../phm_milling_autoenc'.")

In [ ]:

# This cell visualizes the aggregated performance of all autoencoder models.
# It plots the mean, min, and max of metrics (Accuracy, Precision, Recall, F1-Score)
# across all experiments in a single plot for a comprehensive comparison.

if 'autoenc' in output_parent_dir and not results_df.empty:
    
    # --- Determine a common threshold range for all experiments ---
    all_recon_errors = np.concatenate([
        np.array(row['reconstruction_error'], dtype=np.float32) 
        for _, row in results_df.iterrows()
    ])
    
    if all_recon_errors.size > 0:
        thresholds = np.linspace(np.min(all_recon_errors), np.max(all_recon_errors), num=200)

        # --- Calculate metrics for each experiment across the common thresholds ---
        all_metrics = {
            'accuracies': [], 'precisions': [], 'recalls': [], 'f1_scores': []
        }

        for _, row in results_df.iterrows():
            recon_error = np.array(row['reconstruction_error'], dtype=np.float32)
            y_true = np.array(row['y_true'], dtype=np.int32)
            
            # Calculate metrics for the current experiment
            accuracies, precisions, recalls, f1_scores = [], [], [], []
            for t in thresholds:
                y_pred = (recon_error > t).astype(int)
                accuracies.append(accuracy_score(y_true, y_pred))
                precisions.append(precision_score(y_true, y_pred, zero_division=0))
                recalls.append(recall_score(y_true, y_pred, zero_division=0))
                f1_scores.append(f1_score(y_true, y_pred, zero_division=0))
            
            # Store the calculated metrics
            all_metrics['accuracies'].append(accuracies)
            all_metrics['precisions'].append(precisions)
            all_metrics['recalls'].append(recalls)
            all_metrics['f1_scores'].append(f1_scores)

        # --- Aggregate metrics across all experiments ---
        metrics_agg = {
            'Accuracy': {
                'mean': np.mean(all_metrics['accuracies'], axis=0),
                'min': np.min(all_metrics['accuracies'], axis=0),
                'max': np.max(all_metrics['accuracies'], axis=0),
                'color': 'blue'
            },
            'Precision': {
                'mean': np.mean(all_metrics['precisions'], axis=0),
                'min': np.min(all_metrics['precisions'], axis=0),
                'max': np.max(all_metrics['precisions'], axis=0),
                'color': 'orange'
            },
            'Recall': {
                'mean': np.mean(all_metrics['recalls'], axis=0),
                'min': np.min(all_metrics['recalls'], axis=0),
                'max': np.max(all_metrics['recalls'], axis=0),
                'color': 'green'
            },
            'F1-Score': {
                'mean': np.mean(all_metrics['f1_scores'], axis=0),
                'min': np.min(all_metrics['f1_scores'], axis=0),
                'max': np.max(all_metrics['f1_scores'], axis=0),
                'color': 'red'
            }
        }

        # --- Plot the aggregated metrics in one plot ---
        plt.figure(figsize=(16, 16))
        
        for name, data in metrics_agg.items():
            plt.plot(thresholds, data['mean'], label=f'Mean {name}', color=data['color'], linewidth=2 if name == 'F1-Score' else 1)
            plt.fill_between(thresholds, data['min'], data['max'], color=data['color'], alpha=0.1, label=f'{name} Range (Min-Max)')

        plt.title('Aggregated Model Performance vs. Reconstruction Error Threshold', fontsize=16)
        plt.xlabel("Threshold")
        plt.ylabel("Score")
        plt.legend()
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)
        plt.tight_layout()
        plt.show()
    else:
        print("No reconstruction errors found to plot.")
else:
    print("This evaluation is designed for autoencoder models.")
    print("Please set 'output_parent_dir' to an autoencoder experiment directory.")
